Today's topics:
* reading a table with pandas
* column names and column types
* missing values

# Composite tensile strength

A carbon-fibre composite is made of fibres in a resin, and how the fibres are arranged
changes what the part can carry. A woven cloth, a stitched non-crimp fabric and a mat of
chopped fibres can all hold the same weight of fibre and fail at very different loads.

A national testing programme tested 121 specimens of three such architectures in tension,
compression and shear. The dataset records the results.

*What is the tensile strength of this material?* We need a number for the report, and we
get there by reading the file first.

Fibre architecture changes the path that carries an applied load, so these three
conditions should not be treated as interchangeable specimens.

# Tables

For every dataset this semester, we clicked a grey button and two arrays appeared with the
right names in them.

Let's open the button.

In [1]:
import os

import pandas as pd

name = 'composite_mechanical_testing.csv'
url = f'https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/{name}'
local = next((p for p in (f'datasets/{name}', f'../datasets/{name}',
                          f'../../datasets/{name}', f'../../../datasets/{name}')
              if os.path.exists(p)), None)

tests = pd.read_csv(local if local else url)
print(type(tests))

<class 'pandas.core.frame.DataFrame'>


`pd.read_csv` reads a comma-separated file and returns a **DataFrame**: the table itself,
with named columns, one row per specimen, and a type for each column.

It takes a path on disk or a web address. That's why the cell above checks for a local copy
first and then uses the one in the course repository.

The address is worth reading. This file came from a NIST and Department of Energy
composite testing programme, and it is stored in the course repository so the notebook
can access it.

Without a recorded source, the measurements cannot be checked against the original
programme.

In [2]:
print(tests.shape)

(121, 13)


121 rows and 13 columns, and `.shape` reads as it did for a 2-D array in L04: rows first,
then columns.

## Inspection

Let's see what we have.

Printing 121 rows is useless. `.head()` shows the first five:

In [3]:
tests.head()

,condition,test_type,specimen_id,width_mm,thickness_mm,peak_load_N,strength_MPa,modulus_GPa,fracture_toughness_Jm2,peak_engineering_strain_pct,fracture_description,notes,source_file
0,Chopped Fiber 50fvf,Tensile,1-1-5-A-90,24.500000,1.229667,2193.0,73.58,19.902,NaN,NaN,NaN,NaN,ICME-mech-ChoppedProc1-Tensile-AmbientTemp-QSr...
1,Chopped Fiber 50fvf,Tensile,1-1-5-B-90,24.276667,1.388667,1181.0,34.31,21.52,NaN,NaN,NaN,NaN,ICME-mech-ChoppedProc1-Tensile-AmbientTemp-QSr...
2,Chopped Fiber 50fvf,Tensile,1-1-5-C-90,24.280000,1.171333,2434.0,84.91,30.18,NaN,NaN,NaN,NaN,ICME-mech-ChoppedProc1-Tensile-AmbientTemp-QSr...
3,Chopped Fiber 50fvf,Tensile,1-1-5-D-90,24.363333,1.166667,2529.0,88.05,27.66,NaN,NaN,NaN,NaN,ICME-mech-ChoppedProc1-Tensile-AmbientTemp-QSr...
4,Chopped Fiber 50fvf,Tensile,1-1-5-E-90,24.393333,1.306000,3323.0,100.57,28.49,NaN,NaN,NaN,NaN,ICME-mech-ChoppedProc1-Tensile-AmbientTemp-QSr...


Column names across the top, and down the left the index, which numbers the rows.

`.tail()` shows the other end, which is worth a look whenever a file might have a
summary row or a stray blank at the bottom:

In [4]:
tests.tail(3)

,condition,test_type,specimen_id,width_mm,thickness_mm,peak_load_N,strength_MPa,modulus_GPa,fracture_toughness_Jm2,peak_engineering_strain_pct,fracture_description,notes,source_file
118,Twill Weave 660gsm 50fvf,Tensile,Tw-0°-2.3-6,23.53,2.48,NaN,806.7,69.29,NaN,NaN,XGM,NaN,ICME-mech-TwillWeave660gsm50-WarpTensile-Ambie...
119,Twill Weave 660gsm 50fvf,Tensile,Tw-0°-2.3-7,22.97,2.46,NaN,735.7,66.76,NaN,NaN,XGM,NaN,ICME-mech-TwillWeave660gsm50-WarpTensile-Ambie...
120,Twill Weave 660gsm 50fvf,Tensile,Tw-0°-2.3-8,22.94,2.46,NaN,868.1,70.47,NaN,NaN,XGM,Extensometer slippage occurs midway through test,ICME-mech-TwillWeave660gsm50-WarpTensile-Ambie...


The last three rows are ordinary specimens, not a summary row or a blank line.

A spreadsheet provides the same view. A DataFrame also supports computation one column
at a time without opening the file again.

In [5]:
print(tests.columns)

Index(['condition', 'test_type', 'specimen_id', 'width_mm', 'thickness_mm',
       'peak_load_N', 'strength_MPa', 'modulus_GPa', 'fracture_toughness_Jm2',
       'peak_engineering_strain_pct', 'fracture_description', 'notes',
       'source_file'],
      dtype='object')


Thirteen names. `condition` is the fibre architecture, `test_type` is what was done to
the specimen, and the rest are what was measured.

A name is not a definition. What `fracture_toughness_Jm2` means comes from the programme
that recorded it, not from the file.

## Column selection

One name in the brackets gives one column. A **list** of names, the container from L07,
gives several:

In [6]:
tests[['condition', 'test_type']].head()

,condition,test_type
0,Chopped Fiber 50fvf,Tensile
1,Chopped Fiber 50fvf,Tensile
2,Chopped Fiber 50fvf,Tensile
3,Chopped Fiber 50fvf,Tensile
4,Chopped Fiber 50fvf,Tensile


The double brackets are not a special syntax. The outer pair is the indexing we've been
doing since L03, and the inner pair is a list.

## Categories

`condition` is the fibre architecture and `test_type` is what was done to the specimen.
Neither is a number, and counting them is how we find out what the file covers:

In [7]:
print(tests['condition'].value_counts())

condition
Chopped Fiber 50fvf         70
Twill Weave 660gsm 50fvf    37
NCF 50fvf                   14
Name: count, dtype: int64


In [8]:
print(tests['test_type'].value_counts())

test_type
Tensile                       56
Compression                   27
Shear                         11
Off-Axis 45deg Tensile         9
Shear 1-3                      6
Shear 2-3                      5
Fracture Toughness Mode I      4
Fracture Toughness Mode II     3
Name: count, dtype: int64


Three architectures and eight kinds of test, and they're badly unbalanced: 70 of the 121
specimens are chopped fibre, and some test types appear three times.

That matters before any average is taken. A summary over the whole table would weight
chopped fibre heavily and mix eight different kinds of test.

### [Check your understanding]

1. Display the first two rows and the last two rows of `tests`.
2. Display the `condition`, `test_type` and `peak_load_N` columns together.
3. Count the specimens in each `condition` and identify the least represented condition.
4. In a comment, explain whether the three conditions are represented equally.

# Column types

Every column has a type. Pandas inferred each one while reading the file:

In [9]:
print(tests.dtypes)

condition                       object
test_type                       object
specimen_id                     object
width_mm                       float64
thickness_mm                   float64
peak_load_N                    float64
strength_MPa                    object
modulus_GPa                     object
fracture_toughness_Jm2         float64
peak_engineering_strain_pct     object
fracture_description            object
notes                           object
source_file                     object
dtype: object


`float64` for decimals, and `object`, which means text.

Read that list against the column names before going on. `width_mm` and `peak_load_N` are
numbers, as expected. So is `strength_MPa`, which is the number this whole lecture is
about.

It came back as `object`.

## Mixed entries

Something in that column isn't a number. `.value_counts()` counts how often each entry
appears, most common first:

In [10]:
print(tests['strength_MPa'].value_counts().head(3))

strength_MPa
-         11
160.0      2
1109.0     2
Name: count, dtype: int64


Nearly every strength is its own value, appearing once. The most common entry in the
column is a dash, eleven times, and it means the strength wasn't reported for that test.

All rows in a column share one type. Eleven dashes are enough to make all 121 entries text,
including the 110 that are perfectly good numbers.

In [11]:
# EXPECTED-ERROR: this cell fails on purpose -- see the surrounding text
tests['strength_MPa'].median()

TypeError: Cannot convert ['73.58' '34.31' '84.91' '88.05' '100.57' '109.57' nan nan nan '92.57'
 '26.28' '119.61' '120.08' '107.77' '121.78' '40.51' '83.86' '26.36'
 '22.537' '72.19' '48.01' nan '99.1' '121.66' '107.54' '79.38' nan
 '113.85' '68.12' '126.62' '58.66' '79.39' nan '63.11' nan '118.22'
 '84.45' '79.4' nan '97.81' '34.91' '68.06' '105.94' '72.05' '249.21'
 '303.36' '250.63' '254.0' '271.9' '202.56' '173.71' '187.8' '212.3'
 '160.0' '181.23' '150.0' '160.0' '147.27' '169.99' '94.53' '121.53'
 '87.93' '106.15' '76.67' '108.01' '117.31' '120.0' '140.37' '115.04'
 '116.29' '222.2' '226.2' '217.0' '227.2' '1109.0' '1149.0' '1109.0'
 '1200.0' '-' '-' '-' '-' '-' '-' nan nan nan nan nan nan nan '144.8'
 '128.9' '126.2' '126.9' '126.9' '-' '-' '-' '-' '-' '402.7' '394.4'
 '379.9' '413.0' '380.6' '401.3' '466.1' '430.9' '437.1' '430.9' '355.8'
 '433.0' '712.9' '755.6' '680.3' '732.2' '742.4' '806.7' '735.7' '868.1'] to numeric

The median of a column of text is not defined, and pandas raises an error.

We've seen this failure before. In L07 one string in `np.array` turned every number into
text; here one dash in a spreadsheet does the same thing to a column.

## Numeric conversion

In [12]:
strength = pd.to_numeric(tests['strength_MPa'], errors='coerce')
print(strength.dtype)

float64


`pd.to_numeric` converts entries that contain numbers. With `errors='coerce'`, entries
that cannot be converted become missing rather than raising an error.

That is a decision, not a repair. The dashes are gone and 26 entries are now missing.

Our question concerns ordinary tensile tests, not compression, shear or off-axis tensile
tests.

The selection below is a recipe to read today, not syntax to memorize. L10 covers
comparison and selection as syntax.

In [13]:
# Provided recipe: keep the strength values from ordinary tensile tests
tensile_strength = strength[tests['test_type'] == 'Tensile']
print(f'tensile tests: {len(tensile_strength)}')
print(f'recorded strengths: {tensile_strength.notna().sum()}')
print(f'median tensile strength: {tensile_strength.median():.1f} MPa')

tensile tests: 56
recorded strengths: 48
median tensile strength: 99.8 MPa


There are 56 ordinary tensile tests, and 48 have strength recorded. Their median is
99.835 MPa, reported above as 99.8 MPa.

The other test types measure different responses and do not belong in that median.

# Missing values

`NaN` stands for "not a number", and it is what pandas puts where a value is absent.

Missing values were in this table before we made any:

In [14]:
print(tests['peak_load_N'].notna().sum())
print(len(tests))

68
121


68 of the 121 specimens have a peak load recorded and 53 do not. That is not a corner of
the table; it is nearly half of it.

Those tests happened. The load was simply absent from the campaign's recorded numbers.

A value can be missing because the measurement was not made, because it failed, or because
the test was stopped first.

The file records only that it is absent. The programme's documentation is needed to
establish the reason.

## Missing-value arithmetic

In [15]:
import numpy as np

loads = tests['peak_load_N'].to_numpy()
print(np.mean(loads))

nan


`nan`. One missing value is enough to make the mean of the whole array missing, and the
calculation returns that missing result without a warning.

That is the correct answer. The mean of a set of numbers where some are unknown is
unknown.

In [16]:
print(f'{np.nanmean(loads):.0f} N')

3074 N


`np.nanmean` skips them and gives the mean of the 68 recorded loads.

The result is therefore narrower: the average peak load among specimens whose peak load
was recorded.

> Dropping missing values changes the question being answered. It's worth doing
> deliberately, and worth saying how many rows are left.

## Series

A single column is a **Series**, and the statistics from L05 and L06 work on it directly:

In [17]:
print(f"median thickness: {tests['thickness_mm'].median():.2f} mm")
print(f"90th percentile:  {tests['thickness_mm'].quantile(0.9):.2f} mm")

median thickness: 2.47 mm
90th percentile:  4.67 mm


And `.to_numpy()` returns the plain array we have been working with all along:

In [18]:
thickness = tests['thickness_mm'].to_numpy()
print(type(thickness))
print(np.round(thickness[:5], 2))

<class 'numpy.ndarray'>
[1.23 1.39 1.17 1.17 1.31]


The collapsed loaders used the same `.to_numpy()` conversion after reading a file. Until
today, the file-reading step was the part we hadn't covered.

### [Check your understanding]

1. Print the shape of `tests` and say in a comment which number counts specimens.
2. Print how many specimens have a `width_mm` recorded.
3. Take the median of `width_mm` with `.median()`. Convert the column to an array and
   compute both `np.mean` and `np.nanmean`.
4. Compare the three results. In a comment, state which rows each calculation uses.

*Optional challenge: repeat the calculation for `peak_load_N` and report how many values
each usable summary includes.*

## Column summaries

One call summarises every numeric column at once. Nothing in it is new; it's the
statistics from L05 and L06, in a table:

In [19]:
#@title Optional tool glimpse: describe the numeric columns at once (click ▶ to run) { display-mode: "form" }
tests[['width_mm', 'thickness_mm', 'peak_load_N']].describe()

,width_mm,thickness_mm,peak_load_N
count,117.000000,121.000000,68.000000
mean,23.081852,2.380006,3074.213795
std,4.936428,1.445310,2618.245154
min,12.576667,1.138333,1.403000
25%,22.330000,1.306000,504.239750
50%,24.120000,2.470000,3026.500000
75%,25.280000,2.590000,4776.960750
max,31.750000,9.320000,9970.283691


Count, mean, standard deviation, minimum, the quartiles, maximum. Read the count first:
a column with fewer counted values than the others is a column with missing entries.

# Summary

* `pd.read_csv` reads a table from a file or a URL and returns a **DataFrame**.
* A DataFrame preserves column names and one type per column instead of flattening a table.
* `.head()`, `.shape`, `.columns` and `.dtypes` are how we look before we compute.
* A numeric column read as `object` has something in it that isn't a number.
* One column is a **Series**, and `.to_numpy()` gives back the arrays we started with.
* `NaN` marks an absent value. `np.mean` returns `NaN` when an array contains one, while
  `np.nanmean` skips missing entries.

The median tensile strength is 99.8 MPa across the 48 ordinary tensile specimens with a
recorded strength. Eight other ordinary tensile specimens have no strength value in the
table, and the other test types correspond to different mechanical responses.

## Further reading

* VanderPlas, *Python Data Science Handbook*, "Data Manipulation with Pandas"
* The data: NIST and DOE ICME composite mechanical testing programme, DE-EE0006867